# V3 — đo trên mẫu con cbba

Bản V3 của `lexo_do_ban_an.ipynb`. Khác V2: gói đo là `lexo_do_ban_an_cbba.zip` (vụ test + một phần dev), `MO_HINH_LORA` / `MAU_GGUF_LORA` theo
`chon-dieu-cbba-v1`, `CHAY_VONG_1 = False` (gói không có train90), và lượt test chỉ gồm A, C0, L (B0 / B1 tách hành vi quá chậm với cỡ test này).
Cỡ test `N` có thể phải cắt ngắn theo tốc độ Ollama đo ở G4 (quy tắc trong `kaggle/KE_HOACH_KAGGLE_V3.md`): đóng gói lại với `--so-test` nhỏ hơn
cho cùng danh sách test, không tự ý chạy một phần.

# Lexo — đo bộ bản án (Vòng 1, cổng G3/G4 của LoRA chọn Điều, lượt test)

Notebook này chỉ chạy LLM qua Ollama. Sự việc đã cắt, ứng viên đã xếp hạng, mã nguồn và chỉ mục đều nằm sẵn trong gói đo
(sinh trên laptop bằng `scripts/dong_goi_kaggle.py`), nên vector bge-m3 của ứng viên không lệch giữa laptop và Kaggle.
Không cần `lexo_eval_colab.zip` cũ.

**Cài đặt notebook:** Accelerator **GPU T4** (một hay hai đều được), **Internet On**.

**Input (Add Input):**

| Gắn | Khi nào | Notebook tìm theo |
|---|---|---|
| Dataset private từ `lexo_do_ban_an_cbba.zip` | luôn | file `LEXO_DO_BAN_AN.md` |
| Dataset có GGUF n368 | để chạy B1 | tên file chứa `n368` và `q4_k_m`, đuôi `.gguf` |
| Output của notebook `lexo_gop_gguf` | sau Vòng 2 | tên file chứa `chon-dieu-cbba-v1` và `q4_k_m`, đuôi `.gguf` |

**Chạy:** sửa ô «Việc cần chạy» rồi Run All. Lượt dài (Vòng 1 ước 2–3 giờ, lượt test) nên dùng **Save Version → Save & Run All**
để chạy cả khi đóng trình duyệt. Mỗi phần ghi `ket_qua_<tên>.zip` vào Output: tải các zip đó về `inbox/kaggle_ket_qua/` trên laptop.
Kết quả từng vụ ghi vào `.jsonl` ngay sau mỗi vụ; chạy lại ô trong cùng phiên sẽ làm tiếp các vụ còn thiếu.

**Tập test chỉ chạy một lần, ở cuối,** sau khi đã chốt cấu hình ở chat gốc. Ô test bị khóa bằng `DA_CHOT = False`.

In [ ]:
# ---- Việc cần chạy (sửa ở đây rồi Run All) ----
CHAY_VONG_1 = False       # V3: gói cbba không có tập train90 nên không có Vòng 1 (đừng bật)
CHAY_LORA_DEV = False     # sau Vòng 2: cổng G3/G4 của LoRA chọn Điều trên dev (A, C0, L)
CHAY_DOI_K = False        # Vòng 3, C3: đổi K lúc chạy, chỉ trên dev
DOI_K = (10, 30)
CHAY_TEST = False         # lượt test duy nhất: chỉ bật sau khi đã chốt (xem ô cuối)

MO_HINH_N368 = "qwen3-8b-lexo-sach-n368"           # tên trong Ollama (phải chứa "qwen3")
MAU_GGUF_N368 = "*n368*q4_k_m*.gguf"
MO_HINH_LORA = "qwen3-8b-lexo-chon-dieu-cbba-v1"
MAU_GGUF_LORA = "*chon-dieu-cbba-v1*q4_k_m*.gguf"
OLLAMA_VERSION = "0.40.4"  # như các lần đo dev 34 câu; "" = bản mới nhất. Bản thật được ghi vào moi_truong.json
NUM_CTX = 8192
REVISION_QWEN3 = "b968826d9c46dd6066d109eabc6255188de91218"   # chỉ để đếm token bằng tokenizer, cùng bản lúc học

In [ ]:
# ---- Hàm dùng chung (không cần sửa) ----
import collections, fnmatch, hashlib, json, math, os, re, shutil, subprocess, sys, time, zipfile
from collections import Counter
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
TAM = Path("/kaggle/tmp") if Path("/kaggle/tmp").is_dir() else Path("/tmp")   # thư mục tạm lớn, không vào Output


def sha256(p, khoi=1 << 24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(khoi), b""):
            h.update(b)
    return h.hexdigest()


def tim(mau, goc=INPUT):
    """Mọi file có tên khớp mẫu (không phân biệt hoa thường) dưới goc, đi theo cả symlink."""
    ra = []
    for d, _, ten in os.walk(goc, followlinks=True):
        ra += [Path(d) / t for t in ten if fnmatch.fnmatch(t.lower(), mau.lower())]
    return sorted(ra)


def tim_goi(dau, mau_zip):
    """Thư mục gói (chứa file đánh dấu). Kaggle thường tự giải nén zip; nếu không thì giải nén vào thư mục tạm."""
    thay = tim(dau)
    if not thay:
        for z in tim(mau_zip):
            dich = TAM / "giai_nen" / z.stem
            with zipfile.ZipFile(z) as f:
                f.extractall(dich)
            thay = tim(dau, dich)
            break
    if len(thay) != 1:
        raise SystemExit(f"Cần đúng một gói có {dau} trong /kaggle/input, thấy {len(thay)}: {thay}")
    return thay[0].parent


def kiem_sha256sums(goc):
    r = subprocess.run(["sha256sum", "-c", "--quiet", "SHA256SUMS"], cwd=goc, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("SHA256SUMS SAI, dừng lại (tải lại gói):\n" + r.stdout[-3000:] + r.stderr[-1000:])
    n = len([x for x in (goc / "SHA256SUMS").read_text(encoding="utf-8").splitlines() if x.strip()])
    print(f"SHA256SUMS: OK cả {n} file ({goc})")


def chay(lenh, cwd=None, env=None, duoi=None):
    """Chạy lệnh và in đầu ra từng dòng (đầu ra của subprocess thường không hiện trong ô). duoi=N: chỉ in N dòng cuối.
    Lệnh lỗi thì dừng ô."""
    lenh = [str(x) for x in lenh]
    print("$", " ".join(lenh), flush=True)
    p = subprocess.Popen(lenh, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace")
    cuoi = collections.deque(maxlen=duoi or 1)
    for dong in p.stdout:
        if duoi:
            cuoi.append(dong)
        else:
            print(dong, end="", flush=True)
    if duoi:
        print("".join(cuoi), end="", flush=True)
    if p.wait():
        raise RuntimeError(f"lệnh thoát mã {p.returncode}: {' '.join(lenh[:4])} …")


def doc_jsonl(p):
    p = Path(p)
    return [json.loads(x) for x in p.read_text(encoding="utf-8").splitlines() if x.strip()] if p.exists() else []

## 1. Gói đo: kiểm SHA256, chép sang `/kaggle/working/lexo`

In [ ]:
GOI = tim_goi("LEXO_DO_BAN_AN.md", "lexo_do_ban_an*.zip")
kiem_sha256sums(GOI)
LEXO = WORK / "lexo"
if LEXO.exists():
    if (LEXO / "SHA256SUMS").read_bytes() != (GOI / "SHA256SUMS").read_bytes():
        raise SystemExit(f"{LEXO} là bản của một gói khác: xóa thư mục đó (hoặc mở phiên mới) rồi chạy lại")
    print("Đã có", LEXO, "của cùng gói: giữ nguyên để chạy tiếp")
else:
    shutil.copytree(GOI, LEXO)
CAU_HINH = json.loads((LEXO / "cau_hinh.json").read_text(encoding="utf-8"))
if CAU_HINH.get("gia"):
    raise SystemExit("Gói này dựng bằng vector giả (--gia): không dùng để đo. Đóng gói lại trên laptop, không có --gia.")
DU_LIEU = json.loads((LEXO / "ban_an_do.json").read_text(encoding="utf-8"))
K, CHE_DO = CAU_HINH["ung_vien"]["k"], CAU_HINH["ung_vien"]["che_do"]
print("Vụ theo tập:", dict(Counter(x["tap"] for x in DU_LIEU["mau"])))
print(f"Ứng viên: {CHE_DO}, K = {K}; lời nhắc {CAU_HINH['phien_ban_tin_nhan']}; nhúng {CAU_HINH['mo_hinh_nhung']}")
sys.path.insert(0, str(LEXO))
from src.ban_an import chon_dieu as cd                                # cùng mã đã sinh gói
from src.index.tach_hanh_vi import dung_tin_nhan as tin_nhan_tach
assert cd.HE_THONG_CHON == CAU_HINH["he_thong_chon"]
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

## 2. Ollama: cài, chạy nền, kéo `qwen3:8b` và `bge-m3`

`OLLAMA_NUM_PARALLEL=1` để mỗi lần gọi có trọn `num_ctx` (bộ nhớ KV không bị chia). Mô hình lưu ở thư mục tạm, không vào Output.

In [ ]:
import urllib.request

LOG_OLLAMA = WORK / "ollama_serve.log"
URL = "http://localhost:11434"


def ollama_song():
    try:
        with urllib.request.urlopen(URL + "/api/version", timeout=5) as r:
            return json.loads(r.read())["version"]
    except OSError:
        return None


def mo_hinh_co():
    with urllib.request.urlopen(URL + "/api/tags", timeout=30) as r:
        return {m["name"]: m for m in json.loads(r.read())["models"]}


if not shutil.which("ollama"):
    chay(["bash", "-c", "apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y zstd pciutils lshw"], duoi=3)
    chay(["bash", "-c", "curl -fsSL https://ollama.com/install.sh | " + (f"OLLAMA_VERSION={OLLAMA_VERSION} " if OLLAMA_VERSION else "") + "sh"], duoi=5)
if not ollama_song():
    env = dict(os.environ, OLLAMA_MODELS=str(TAM / "ollama_models"), OLLAMA_NUM_PARALLEL="1", OLLAMA_MAX_LOADED_MODELS="2",
               OLLAMA_KEEP_ALIVE="30m")
    subprocess.Popen(["ollama", "serve"], stdout=open(LOG_OLLAMA, "a"), stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for _ in range(90):
        time.sleep(1)
        if ollama_song():
            break
PHIEN_BAN_OLLAMA = ollama_song()
assert PHIEN_BAN_OLLAMA, f"Ollama không chạy: xem {LOG_OLLAMA}"
print("Ollama", PHIEN_BAN_OLLAMA)
for m in ("qwen3:8b", "bge-m3"):
    chay(["ollama", "pull", m], duoi=2)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 3. Mô hình từ GGUF: n368 (cho B1) và LoRA chọn Điều (cho L)

Modelfile lấy nguyên khuôn chat và tham số của `qwen3:8b` (`ollama show qwen3:8b --modelfile`), chỉ thay dòng `FROM`, như các lần trước.

In [ ]:
GGUF = {}   # tên Ollama -> file, sha256, kích thước


def tao_tu_gguf(ten, mau):
    ds = tim(mau)
    if not ds:
        print(f"Không thấy GGUF khớp {mau!r} trong /kaggle/input: chưa có {ten}")
        return False
    if len(ds) > 1:
        raise SystemExit(f"Nhiều GGUF khớp {mau!r}: {ds}")
    print(f"{ten}: tính sha256 của {ds[0].name} …", flush=True)
    GGUF[ten] = {"file": ds[0].name, "sha256": sha256(ds[0]), "bytes": ds[0].stat().st_size}
    print("  sha256", GGUF[ten]["sha256"])
    if any(t in (ten, ten + ":latest") for t in mo_hinh_co()):
        print(f"  {ten} đã có trong Ollama")
        return True
    mf = subprocess.run(["ollama", "show", "qwen3:8b", "--modelfile"], capture_output=True, text=True, check=True).stdout
    mf, n = re.subn(r"(?m)^FROM .*$", f"FROM {ds[0]}", mf, count=1)
    assert n == 1, "không thấy dòng FROM trong Modelfile của qwen3:8b"
    p = TAM / f"Modelfile_{ten}"
    p.write_text(mf, encoding="utf-8")
    chay(["ollama", "create", ten, "-f", p], duoi=3)
    return True


CO_N368 = tao_tu_gguf(MO_HINH_N368, MAU_GGUF_N368)
CO_LORA = tao_tu_gguf(MO_HINH_LORA, MAU_GGUF_LORA)

## 4. Hàm kiểm kỹ thuật (G3/G4) và đóng gói kết quả

- Số vụ lỗi, có khối think, «có thể bị cắt» (prompt_eval_count sát num_ctx), phải bằng 0.
- Nhật ký Ollama không được có dòng «truncating» (Ollama cắt ngầm khi ngữ cảnh quá nhỏ).
- `prompt_eval_count` so với số token tự đếm bằng tokenizer Qwen3 (cùng revision lúc học). Lệch vài token là do khuôn chat của
  Ollama; lệch lớn thì xem lại Modelfile.
- `ollama ps` phải báo `100% GPU`.

In [ ]:
from IPython.display import Markdown, display

try:
    from transformers import AutoTokenizer
    TOK = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B", revision=REVISION_QWEN3)
except Exception as e:                                           # không chặn việc đo nếu không tải được tokenizer
    TOK = None
    print("CHÚ Ý: không tải được tokenizer Qwen3, bỏ phần so số token:", e)
THEO_ID = {x["id"]: x for x in DU_LIEU["mau"]}


def dem_token(c, r):
    x = THEO_ID[r["id"]]
    msgs = tin_nhan_tach(x["su_viec"], "thuan") if c in ("B0", "B1") else cd.dung_tin_nhan(x["su_viec"], r["ung_vien"], x["tieu_de"])
    s = TOK.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return len(TOK(s, add_special_tokens=False)["input_ids"])


def kiem_log():
    chu = LOG_OLLAMA.read_text(encoding="utf-8", errors="replace") if LOG_OLLAMA.exists() else ""
    cat = [d for d in chu.splitlines() if "truncat" in d.lower()]
    (WORK / "ollama_canh_bao.txt").write_text("\n".join(cat) + "\n", encoding="utf-8")
    print(f"Nhật ký Ollama: {len(cat)} dòng báo cắt ngữ cảnh" + (" → KHÔNG ĐẠT (xem ollama_canh_bao.txt)" if cat else " → đạt"))
    return not cat


def kiem_gpu():
    ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
    print(ps)
    return "CPU" not in ps


def kiem_ky_thuat(thu_muc, cac_cau_hinh):
    dat = True
    for c in cac_cau_hinh:
        ds = doc_jsonl(Path(thu_muc) / f"{c}.jsonl")
        if not ds:
            print(f"{c}: chưa có kết quả")
            dat = False
            continue
        loi, think, cat = (sum(bool(r.get(t)) for r in ds) for t in ("loi", "co_think", "co_the_bi_cat"))
        dong = f"{c}: {len(ds)} vụ | lỗi {loi} | khối think {think} | có thể bị cắt {cat}"
        if TOK is not None:
            lech = [dem_token(c, r) - r["prompt_eval_count"] for r in ds if r.get("prompt_eval_count") is not None]
            if lech:
                dong += f" | token tự đếm − prompt_eval_count: từ {min(lech)} đến {max(lech)}" + (" (khớp)" if max(map(abs, lech)) <= 8 else " (LỆCH)")
        print(dong)
        for r in ds:
            if r.get("loi"):
                print(f"   {r['id']}: {r['loi']}")
        dat = dat and not (loi or think or cat)
    return dat


def chay_do(*tham_so):
    chay([sys.executable, "-u", "scripts/do_ban_an.py", "--num-ctx", NUM_CTX, *tham_so], cwd=LEXO)


def bao_cao(thu_muc):
    display(Markdown((Path(thu_muc) / "bao_cao.md").read_text(encoding="utf-8")))


def ghi_moi_truong():
    co = mo_hinh_co()
    mt = {"ollama": PHIEN_BAN_OLLAMA, "num_ctx": NUM_CTX,
          "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True,
                                text=True).stdout.strip().splitlines(),
          "mo_hinh": {t: {"digest": m.get("digest"), "size": m.get("size")} for t, m in co.items()}, "gguf": GGUF,
          "goi_do": {"sha256_SHA256SUMS": sha256(LEXO / "SHA256SUMS"), "ung_vien": CAU_HINH["ung_vien"],
                     "phien_ban_tin_nhan": CAU_HINH["phien_ban_tin_nhan"]},
          "luc": time.strftime("%Y-%m-%d %H:%M:%S")}
    (WORK / "moi_truong.json").write_text(json.dumps(mt, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    return mt


def dong_goi(nhan, cac):
    z = WORK / f"ket_qua_{nhan}.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f:
        for p in [*map(Path, cac), WORK / "moi_truong.json", WORK / "ollama_canh_bao.txt"]:
            if p.is_dir():
                for q in sorted(p.rglob("*")):
                    if q.is_file():
                        f.write(q, q.relative_to(p.parent))
            elif p.exists():
                f.write(p, p.name)
    print(f"Đã ghi {z} ({z.stat().st_size / 1e6:.1f} MB): tải về inbox/kaggle_ket_qua/")

## 5. Vòng 1: A, B0, B1, C0 trên 90 vụ train

Chạy thử 3 vụ trước (thư mục riêng `thu_vong_1`, không lẫn vào kết quả chính); có lỗi kỹ thuật thì dừng trước lượt dài.
Cách đọc bảng (ghi trước khi có số) nằm trong tài liệu kế hoạch, mục Vòng 1: quyết định sang Vòng 2 làm ở chat gốc.

In [ ]:
if CHAY_VONG_1:
    ghi_moi_truong()
    cau_hinh = ["A", "B0"] + (["B1"] if CO_N368 else []) + ["C0"]
    if not CO_N368:
        print("CHÚ Ý: không có GGUF n368 → bỏ B1 (so sánh B1 với B0 sẽ thiếu)")
    llm = [c for c in cau_hinh if c != "A"]
    train90 = [x for x in DU_LIEU["mau"] if x["tap"] in ("hoc", "dev")]
    ba = list(dict.fromkeys([max(train90, key=lambda x: x["so_tu_goc"])["id"],
                             next(x["id"] for x in train90 if len(x["dieu"]) > 1), train90[0]["id"]]))
    THU1 = WORK / "thu_vong_1"
    chay_do("--tap", "train90", "--cau-hinh", ",".join(llm), "--chi-id", ",".join(ba), "--ra", THU1)
    ok = kiem_gpu() & kiem_ky_thuat(THU1, llm) & kiem_log()
    if not ok:
        dong_goi("thu_vong_1", [THU1])
        raise SystemExit("Chạy thử 3 vụ chưa sạch về kỹ thuật: dừng trước lượt dài, mang ket_qua_thu_vong_1.zip về chat gốc")
    chay_do("--tap", "train90", "--cau-hinh", ",".join(cau_hinh))
    RA1 = LEXO / "ket_qua" / "train90"
    kiem_ky_thuat(RA1, llm)
    kiem_log()
    bao_cao(RA1)
    dong_goi("train90", [RA1, THU1])

## 6. Sau Vòng 2: G3 và G4 của LoRA chọn Điều trên dev

Cần gắn output của `lexo_gop_gguf`. G3: 3 vụ dev, mô hình nằm trên GPU, `num_ctx` 8192, không lỗi, số token khớp.
G4: cả các vụ dev chạy không lỗi. Điểm tìm đủ / khớp đúng trên dev chỉ để xem, **không dùng để chọn mô hình**.

In [ ]:
if CHAY_LORA_DEV:
    if not CO_LORA:
        raise SystemExit(f"Chưa gắn GGUF khớp {MAU_GGUF_LORA} (output của notebook lexo_gop_gguf)")
    ghi_moi_truong()
    dev = sorted(x["id"] for x in DU_LIEU["mau"] if x["tap"] == "dev")
    THU3 = WORK / f"thu_g3_{MO_HINH_LORA}"
    chay_do("--tap", "dev", "--cau-hinh", "C0,L", "--mo-hinh-chon", MO_HINH_LORA, "--chi-id", ",".join(dev[:3]), "--ra", THU3)
    g3 = kiem_gpu() & kiem_ky_thuat(THU3, ["C0", "L"]) & kiem_log()
    print("G3:", "ĐẠT" if g3 else "KHÔNG ĐẠT")
    if not g3:
        dong_goi(f"g3_{MO_HINH_LORA}", [THU3])
        raise SystemExit("G3 không đạt: dừng, mang zip về chat gốc")
    RA2 = LEXO / "ket_qua" / f"dev_{MO_HINH_LORA}"
    chay_do("--tap", "dev", "--cau-hinh", "A,C0,L", "--mo-hinh-chon", MO_HINH_LORA, "--ra", RA2)
    g4 = kiem_ky_thuat(RA2, ["C0", "L"]) & kiem_log()
    print(f"G4 ({len(dev)} vụ dev):", "ĐẠT" if g4 else "KHÔNG ĐẠT")
    bao_cao(RA2)
    dong_goi(f"dev_{MO_HINH_LORA}", [RA2, THU3])

## 7. Vòng 3, C3 (tùy chọn): đổi K lúc chạy, chỉ trên dev

Không học lại. Phải xong trước lượt test.

In [ ]:
if CHAY_DOI_K:
    ghi_moi_truong()
    cac = []
    for k in DOI_K:
        ra = LEXO / "ket_qua" / f"dev_k{k}"
        chay_do("--tap", "dev", "--cau-hinh", "A,C0" + (",L" if CO_LORA else ""), *(["--mo-hinh-chon", MO_HINH_LORA] if CO_LORA else []),
                "--k", k, "--ra", ra)
        kiem_ky_thuat(ra, ["C0"] + (["L"] if CO_LORA else []))
        bao_cao(ra)
        cac.append(ra)
    kiem_log()
    dong_goi("dev_doi_k", cac)

## 8. Lượt test: một lần duy nhất

Chỉ làm khi: Vòng 2 (và Vòng 3 nếu làm) đã xong, cấu hình đã chốt ở chat gốc dựa trên val loss, K chọn trên học + dev và cổng kỹ thuật G3 / G4.
Điền `CHOT_CHON`, đặt `DA_CHOT = True` và `CHAY_TEST = True` (tắt các phần khác), rồi **Save Version → Save & Run All**.

Ô này ghi `cau_hinh_chot.json` (mô hình, SHA256 của GGUF, digest trong Ollama, K, chế độ ứng viên, lời nhắc, `num_ctx`,
phiên bản Ollama) rồi gọi `do_ban_an.py --tap test --da-chot --chot`. Script lấy mọi tham số từ file chốt, ghi sha256 của file
vào từng dòng kết quả và từ chối chạy tiếp nếu file chốt khác. Sau lượt test: chép bảng số tổng vào tài liệu trước, rồi mới xem từng vụ.

In [ ]:
DA_CHOT = False
CHOT_CHON = {
    "cau_hinh": ["A", "C0", "L"],               # V3: truy hồi, mô hình gốc, LoRA cbba; B0 / B1 (tách hành vi) quá chậm với cỡ test này
    "mo_hinh_goc": "qwen3:8b",
    "mo_hinh_tach": MO_HINH_N368,
    "mo_hinh_chon": MO_HINH_LORA,
    "k": K,                                     # K đã chọn trên 90 vụ train (hoặc K khác nếu C3 trên dev cho lý do rõ)
    "che_do_uv": CHE_DO,
    "num_ctx": NUM_CTX,
}

if CHAY_TEST:
    if not DA_CHOT:
        raise SystemExit("Chưa chốt: điền CHOT_CHON theo quyết định ở chat gốc rồi đặt DA_CHOT = True")
    can = {"B1": CHOT_CHON["mo_hinh_tach"], "L": CHOT_CHON["mo_hinh_chon"]}
    thieu = [m for c, m in can.items() if c in CHOT_CHON["cau_hinh"] and m not in GGUF]
    if thieu:
        raise SystemExit(f"Chưa gắn GGUF cho {thieu}")
    mt = ghi_moi_truong()
    dung = [CHOT_CHON["mo_hinh_goc"]] + [m for c, m in can.items() if c in CHOT_CHON["cau_hinh"]]
    chot = dict(CHOT_CHON, phien_ban_ollama=PHIEN_BAN_OLLAMA, phien_ban_tin_nhan=CAU_HINH["phien_ban_tin_nhan"],
                sha256_he_thong_chon=hashlib.sha256(CAU_HINH["he_thong_chon"].encode("utf-8")).hexdigest(),
                gguf={m: GGUF[m] for m in dung if m in GGUF},
                digest_ollama={m: (mt["mo_hinh"].get(m) or mt["mo_hinh"].get(m + ":latest") or {}).get("digest") for m in dung},
                sha256_ban_an_do=CAU_HINH["sha256"]["ban_an_do.json"])
    F_CHOT = WORK / "cau_hinh_chot.json"
    if F_CHOT.exists():
        if json.loads(F_CHOT.read_text(encoding="utf-8")) != chot:
            raise SystemExit("cau_hinh_chot.json đã có và khác cấu hình này: test chỉ chạy một lần, không chạy lại với cấu hình khác")
    else:
        F_CHOT.write_text(json.dumps(chot, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    print("sha256 file chốt:", sha256(F_CHOT))
    chay([sys.executable, "-u", "scripts/do_ban_an.py", "--tap", "test", "--da-chot", "--chot", F_CHOT], cwd=LEXO)
    RAT = LEXO / "ket_qua" / "test"
    kiem_ky_thuat(RAT, [c for c in chot["cau_hinh"] if c != "A"])
    kiem_log()
    bao_cao(RAT)
    dong_goi("test", [RAT, F_CHOT])